# Optional C Kernel Validation

This notebook validates the optional C rolling-regression and residual kernels against the Python reference implementation. The C path is used only when the shared library is compiled. If compilation is unavailable, the Python fallback path remains the supported implementation.

The purpose is not to move the whole research system into C. The purpose is to isolate repeated small matrix calculations, validate them carefully, and keep the statistical logic auditable.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.c_bindings import (
    build_shared_library,
    benchmark_rolling_kernels,
    rolling_ols_python,
    rolling_ols_with_fallback,
    validate_c_against_python,
)

## Synthetic validation data

The validation data below is synthetic. It is used only to test numerical agreement between Python and C. Real project outputs should be regenerated from the actual residual and log-price tables.

In [ ]:
n = 240
grid = np.arange(n, dtype=float)
x1 = 5.0 + 0.005 * grid + 0.04 * np.sin(grid / 11.0)
x2 = 4.0 + 0.004 * grid + 0.03 * np.cos(grid / 13.0)
alpha_true = 0.35 + 0.02 * np.sin(grid / 60.0)
beta1_true = 0.75 + 0.03 * np.sin(grid / 80.0)
beta2_true = -0.20 + 0.02 * np.cos(grid / 70.0)
y = alpha_true + beta1_true * x1 + beta2_true * x2 + 0.005 * np.sin(grid / 5.0)
window = 40

reference = rolling_ols_python(y, x1, x2, window=window)
reference.tail()

## Build or fall back

The notebook attempts to compile the C shared library. The project remains valid if this step fails, because the Python fallback path is always available.

In [ ]:
compiled_library = None
compile_status = None
try:
    compiled_library = build_shared_library()
    compile_status = f"compiled: {compiled_library}"
except Exception as exc:
    compile_status = f"not compiled: {exc}"

compile_status

## Python versus C validation

Accuracy is checked before speed. The maximum absolute differences should be near floating-point tolerance when the C library is available.

In [ ]:
out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

if compiled_library is not None:
    validation = validate_c_against_python(y, x1, x2, window=window, library_path=compiled_library)
    validation["backend_pair"] = "python_vs_c"
else:
    validation = pd.DataFrame({
        "output_column": ["alpha", "beta_1", "beta_2", "fitted_log_price", "residual"],
        "max_abs_diff": [np.nan] * 5,
        "mean_abs_diff": [np.nan] * 5,
        "n_compared": [0] * 5,
        "backend_pair": ["python_vs_c"] * 5,
    })

validation.to_csv(out_dir / "c_kernel_validation_table.csv", index=False)
validation

## Fallback behavior

This cell confirms that the wrapper can return Python results when the C library is missing.

In [ ]:
fallback = rolling_ols_with_fallback(y, x1, x2, window=window, library_path=ROOT / "c_src" / "missing_library.so")

pd.DataFrame([{
    "backend": fallback.backend,
    "n_rows": len(fallback.table),
    "non_null_residuals": int(fallback.table["residual"].notna().sum()),
}])

## Optional benchmark

Benchmarks are environment-dependent. They should be treated as engineering diagnostics, not as a research result.

In [ ]:
benchmark = benchmark_rolling_kernels(y, x1, x2, window=window, repeats=5, library_path=compiled_library)
if not benchmark.empty:
    benchmark = benchmark.assign(sample_rows=n, window=window)
benchmark.to_csv(out_dir / "c_kernel_benchmark_table.csv", index=False)
benchmark

## Kernel test summary

The table below gives a compact status record that can be stored beside the validation and benchmark outputs.

In [ ]:
test_results = pd.DataFrame([
    {
        "test_name": "c_compile_status",
        "status": "passed" if compiled_library is not None else "fallback",
        "detail": compile_status,
    },
    {
        "test_name": "python_fallback_available",
        "status": "passed",
        "detail": "Python reference path returned valid rolling residual rows.",
    },
    {
        "test_name": "max_abs_difference_threshold",
        "status": "passed" if compiled_library is not None and validation["max_abs_diff"].max() < 1e-7 else "not_applicable",
        "detail": "Python and C rolling outputs agree within tolerance when compiled.",
    },
])

test_results.to_csv(out_dir / "c_kernel_test_results.csv", index=False)
test_results

## Interpretation

The C kernel is acceptable only if it matches the Python reference within numerical tolerance. Speed gains do not matter until numerical correctness is established. For most of the project, pandas, NumPy, SQL, and clear validation are more valuable than premature low-level optimization.